In [2]:
import pandas as pd          # manipulación de datos
import numpy as np           # operaciones matemáticas
from sklearn.model_selection import train_test_split  # divide datos en train/test
from sklearn.preprocessing import StandardScaler      # escala los números
from sklearn.decomposition import PCA                 # reducción de dimensionalidad
import joblib                # guarda objetos de Python en disco (el scaler)
import os                    # manejo de carpetas del sistema

df = pd.read_parquet("../data/processed/saberpro_sistemas_clean.parquet")
print(df.shape)

(38109, 62)


# Variable objetivo

In [3]:
score_cols = [
    "mod_razona_cuantitat_punt",
    "mod_comuni_escrita_punt",
    "mod_ingles_punt",
    "mod_lectura_critica_punt",
    "mod_competen_ciudada_punt",
]

df[score_cols] = df[score_cols].apply(pd.to_numeric, errors="coerce") # convierte a número por si quedaron como texto al guardar/cargar el parquet
df["promedio_puntaje"] = df[score_cols].mean(axis=1) # calcula el promedio de los 5 módulos por estudiante (axis=1 = por fila)

# COMENTADO: el objetivo cambia de regresión a clasificación;
# se predice si el estudiante rinde "alto" o "bajo", no el número exacto
# y = df["promedio_puntaje"].copy()
# # y ahora es un número real (ej: 163.4, 142.8, 178.0)
# # el modelo aprenderá a predecir ese número exacto
# print(f"Promedio de puntajes — media:   {y.mean():.2f}")
# print(f"Promedio de puntajes — mediana: {y.median():.2f}")
# print(f"Promedio de puntajes — mín:     {y.min():.2f}")
# print(f"Promedio de puntajes — máx:     {y.max():.2f}")
# print(f"Promedio de puntajes — std:     {y.std():.2f}")

# PARA CLASIFICACIÓN: y = variable binaria (0=bajo, 1=alto)
# target fue creado en data_processing: corte en la mediana del promedio
y = df["target"].copy()
# 0 → promedio < mediana (rendimiento bajo)
# 1 → promedio >= mediana (rendimiento alto)

print("Balance del target:")
print(y.value_counts(normalize=True).round(3))
# debe estar cerca de 50/50 porque se cortó en la mediana

Balance del target:
target
1    0.5
0    0.5
Name: proportion, dtype: float64


# Índice socioeconómico
### El propósito de esta celda es crear un solo número que resuma el nivel socioeconómico de cada estudiante, combinando estrato, educación de padres, posesiones del hogar, etc.

In [7]:
se_cols = [
    "estrato_num",           # estrato 1-6
    "educ_padre_num",        # nivel educativo padre 0-9
    "educ_madre_num",        # nivel educativo madre 0-9
    "matricula_num",         # costo de matrícula 0-7
    "fami_tieneautomovil_bin",   # tiene auto: 0 o 1
    "fami_tienelavadora_bin",    # tiene lavadora: 0 o 1
    "fami_tienecomputador_bin",  # tiene computador: 0 o 1
    "fami_tieneinternet_bin",    # tiene internet: 0 o 1
]
# estas 8 variables juntas describen el nivel socioeconómico del estudiante

se_cols = [c for c in se_cols if c in df.columns]
# filtro de seguridad: solo usa las que existen en df

df_se = df[se_cols].copy().fillna(df[se_cols].median())
# copia solo esas 8 columnas
# rellena NaN con la mediana de cada columna antes de calcular el índice

# NOTA: el índice SE y el PCA se calculan DESPUÉS del split (ver celda siguiente)
# Aquí solo guardamos df_se para usarlo después
# Por qué: si calculamos min/max/PCA aquí usamos datos del test → trampa
print("df_se listo. El índice SE se calculará después del split.")
print(f"Correlación estrato con promedio_puntaje: {df['estrato_num'].corr(df['promedio_puntaje']):.3f}")

# ── CORRECCIÓN DATA LEAKAGE ───────────────────────────────────────────────────
# Necesitamos los índices train antes de calcular min/max/PCA
# Usamos la misma semilla que el split definitivo para que coincidan
from sklearn.model_selection import train_test_split as tts
idx_train, idx_test = tts(df.index.tolist(), test_size=0.2, random_state=42, stratify=y)

df_se_train = df_se.loc[idx_train]

# min/max SOLO del train
se_min = df_se_train.min()
se_max = df_se_train.max()
rango  = (se_max - se_min).replace(0, 1)

# Normalizar TODO el df con parámetros del train
df["indice_se"] = ((df_se - se_min) / rango).mean(axis=1)

# PCA SOLO del train
df_se_std_train = (df_se_train - df_se_train.mean()) / df_se_train.std().replace(0, 1)
df_se_std_all   = (df_se      - df_se_train.mean()) / df_se_train.std().replace(0, 1)

pca = PCA(n_components=1)
pca.fit(df_se_std_train)
df["indice_se_pca"] = pca.transform(df_se_std_all)

print(f"Varianza explicada PCA-SE (solo train): {pca.explained_variance_ratio_[0]:.2%}")
print(df[["indice_se", "indice_se_pca", "promedio_puntaje"]].corr().round(3))

df_se listo. El índice SE se calculará después del split.
Correlación estrato con promedio_puntaje: 0.281
Varianza explicada PCA-SE (solo train): 30.77%
                  indice_se  indice_se_pca  promedio_puntaje
indice_se             1.000          0.979             0.285
indice_se_pca         0.979          1.000             0.314
promedio_puntaje      0.285          0.314             1.000


# Los dos experimentos

In [8]:
# Experimento 1: SOLO variables socioeconómicas
# Pregunta que responde: ¿pueden las variables SE predecir el rendimiento SOLAS?
# Si el modelo funciona bien con solo estas variables
features_socioeconomicas = [
    "estrato_num",                   # estrato del hogar
    "educ_padre_num",                # nivel educativo padre
    "educ_madre_num",                # nivel educativo madre
    "matricula_num",                 # costo de matrícula (proxy de calidad univ.)
    "horas_trabajo_num",             # horas que trabaja por semana
    "fami_tieneautomovil_bin",       # posesiones del hogar
    "fami_tienelavadora_bin",
    "fami_tienecomputador_bin",
    "fami_tieneinternet_bin",
    "estu_pagomatriculabeca_bin",    # cómo financia los estudios
    "estu_pagomatriculacredito_bin",
    "estu_pagomatriculapadres_bin",
    "estu_pagomatriculapropio_bin",
    "indice_se",                     # el índice compuesto que calculamos arriba
]

# Experimento 2: TODAS las variables
# Pregunta: ¿cuál es el mejor modelo posible con toda la información disponible?
# Comparar exp1 vs exp2 muestra exactamente cuánto aporta la info institucional

cols_excluir = score_cols + ["promedio_puntaje", "periodo", "ingles_desem_num", "comunica_desem_num", "mod_razona_cuantitat_punt", "mod_comuni_escrita_punt", "mod_comuni_escrita_desem", "mod_ingles_desem", "mod_lectura_critica_punt", "mod_ingles_punt", "mod_competen_ciudada_punt"]
# excluimos los puntajes individuales y el promedio porque SON el target
# si los incluimos el modelo "hace trampa": predice el promedio usando el promedio
# periodo también se excluye: es el año del examen, no dice nada del estudiante

# COMENTADO: antes se excluía condicionalmente porque target era opcional en regresión;
# ahora target ES el y, siempre debe excluirse para evitar data leakage
# if "target" in df.columns:
#     cols_excluir.append("target")

# PARA CLASIFICACIÓN: target siempre va en cols_excluir
cols_excluir.append("target")
# si target aparece como feature el modelo aprendería la respuesta directamente

features_todas = [
    c for c in df.columns          # recorre todas las columnas del dataframe
    if c not in cols_excluir       # descarta las excluidas
    and df[c].dtype in ["float64", "int64", "int32", "float32"]
    # solo columnas numéricas (las que codificamos en data_processing)
    # las columnas de texto original se descartan automáticamente
]

print(f"Features experimento 1 (solo SE): {len(features_socioeconomicas)}")
print(f"Features experimento 2 (todas):   {len(features_todas)}")
print(f"\nFeatures exp2:\n{features_todas}")
# imprime la lista completa para verificar que no entró nada raro

Features experimento 1 (solo SE): 14
Features experimento 2 (todas):   26

Features exp2:
['estrato_num', 'educ_padre_num', 'educ_madre_num', 'horas_trabajo_num', 'matricula_num', 'fami_tieneautomovil_bin', 'fami_tienelavadora_bin', 'fami_tienecomputador_bin', 'fami_tieneinternet_bin', 'estu_pagomatriculabeca_bin', 'estu_pagomatriculacredito_bin', 'estu_pagomatriculapadres_bin', 'estu_pagomatriculapropio_bin', 'estu_genero_bin', 'inst_caracter_academico_INSTITUCIÓN UNIVERSITARIA', 'inst_caracter_academico_TÉCNICA PROFESIONAL', 'inst_caracter_academico_UNIVERSIDAD', 'inst_origen_NO OFICIAL - FUNDACIÓN', 'inst_origen_OFICIAL DEPARTAMENTAL', 'inst_origen_OFICIAL MUNICIPAL', 'inst_origen_OFICIAL NACIONAL', 'inst_origen_REGIMEN ESPECIAL', 'estu_metodo_prgm_DISTANCIA VITUAL', 'estu_metodo_prgm_PRESENCIAL', 'indice_se', 'indice_se_pca']


# Split + Scaler + Guardar

In [9]:
# COMENTADO: y era df["promedio_puntaje"] en regresión; ahora es df["target"] (binaria)
# # y ya está definida arriba como df["promedio_puntaje"]

# y está definida arriba como df["target"] — variable binaria 0/1

for exp_name, feat_list in [("exp1_se", features_socioeconomicas),
                             ("exp2_todas", features_todas)]:
# itera dos veces: una para cada experimento
# exp_name → "exp1_se" o "exp2_todas" (nombre para los archivos)
# feat_list → la lista de features de ese experimento

    feat_list = [c for c in feat_list if c in df.columns]
    # filtro de seguridad: elimina features que no existan en df

    X = df[feat_list].fillna(df[feat_list].median()).copy()
    # X = matriz de features para este experimento
    # .fillna(median) último relleno de NaN residuales
    # .copy() → copia independiente para no modificar df

    # SPLIT
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,      # 20% para test, 80% para train
        random_state=42,    # semilla fija siempre la misma división
                            # importante para reproducibilidad del paper
        stratify=y,         # garantiza la misma proporción 0/1 en train y test
    )

    # SCALER
    scaler = StandardScaler() # fórmula: valor_escalado = (valor_original - media) / std
    # resultado: todos los valores quedan centrados en 0
    # valores debajo de la media → negativos
    # valores encima de la media → positivos
    # la unidad ya no son "estratos" sino "desviaciones estándar"
    # StandardScaler estandariza cada columna para que tenga:
    # media = 0 y desviación estándar = 1
    # por qué: estrato va de 0-6, matricula de 0-7, internet de 0-1
    # sin escalar, el modelo pensaría que matricula (0-7) importa más
    # que internet (0-1) solo por su escala numérica
    # con escalar todas tienen el mismo peso inicial

    X_train_sc = pd.DataFrame(
        scaler.fit_transform(X_train),
        # fit_transform en TRAIN:
        # fit calcula la media y std de cada columna CON DATOS DE TRAIN
        # transform → aplica la estandarización
        columns=feat_list
        # conserva los nombres de columnas
    )

    X_test_sc = pd.DataFrame(
        scaler.transform(X_test),
        # transform (sin fit) en TEST:
        # usa la media y std aprendidas del TRAIN, no recalcula
        # por qué: en producción real no conoces el test de antemano
        # si calcularas la media del test estarías usando info del futuro
        columns=feat_list
    )

    #  GUARDAR
    X_train_sc.to_parquet(f"../data/processed/X_train_{exp_name}.parquet", index=False)
    X_test_sc.to_parquet(f"../data/processed/X_test_{exp_name}.parquet",   index=False)
    # guarda las matrices de features escaladas

    y_train.reset_index(drop=True).to_frame().to_parquet(
        f"../data/processed/y_train_{exp_name}.parquet", index=False)
    y_test.reset_index(drop=True).to_frame().to_parquet(
        f"../data/processed/y_test_{exp_name}.parquet",  index=False)
    # reset_index(drop=True) → reinicia el índice de 0 a N
    # necesario porque train_test_split mantiene los índices originales
    # (ej: filas 5,8,12,31...) y al guardar causaría inconsistencias
    # to_frame() → convierte Serie a DataFrame para poder usar to_parquet

    joblib.dump(scaler, f"../data/processed/scaler_{exp_name}.pkl")
    # guarda el scaler entrenado en disco
    # lo necesitas en el futuro para escalar datos de nuevos estudiantes
    # con los mismos parámetros (misma media y std del train)

    print(f"\n{exp_name} — Train: {X_train_sc.shape} | Test: {X_test_sc.shape}")
    # shape (filas, columnas): cuántos estudiantes y features en cada parte

    # COMENTADO: con regresión se verificaba la media del puntaje continuo;
    # para clasificación verificamos el balance de clases entre train y test
    # print(f"  y_train — media: {y_train.mean():.2f} | std: {y_train.std():.2f}")
    # print(f"  y_test  — media: {y_test.mean():.2f}  | std: {y_test.std():.2f}")

    # PARA CLASIFICACIÓN: balance 0/1 debe ser similar en train y test
    print(f"  y_train: {y_train.value_counts(normalize=True).round(3).to_dict()}")
    print(f"  y_test:  {y_test.value_counts(normalize=True).round(3).to_dict()}")
    # si stratify=y funcionó ambos deben mostrar proporciones casi idénticas

print("\nArtefactos guardados en data/processed/")


exp1_se — Train: (30487, 14) | Test: (7622, 14)
  y_train: {1: 0.5, 0: 0.5}
  y_test:  {1: 0.5, 0: 0.5}

exp2_todas — Train: (30487, 26) | Test: (7622, 26)
  y_train: {1: 0.5, 0: 0.5}
  y_test:  {1: 0.5, 0: 0.5}

Artefactos guardados en data/processed/


# Gráficas 

Al final de este notebook quedan guardados en `data/processed/` exactamente estos archivos:
```
X_train_exp1_se.parquet    → features SE escaladas para entrenar
X_test_exp1_se.parquet     → features SE escaladas para evaluar
X_train_exp2_todas.parquet → todas las features escaladas para entrenar
X_test_exp2_todas.parquet  → todas las features escaladas para evaluar
y_train_exp1_se.parquet    → promedios de puntaje para entrenar (números reales)
y_test_exp1_se.parquet     → promedios de puntaje para evaluar
scaler_exp1_se.pkl         → el scaler guardado (para usar con nuevos estudiantes)
scaler_exp2_todas.pkl